## Actividad 3_17: Regresión con RN. ¿Cuanto vale un portátil?
<div style="border-style:groove;border-width:thin;padding:10px">
Ya hemos hecho un ejercicio de clasificación con redes neuronales. Vamos a trabajar ahora con uno de regresión. En este caso es un dataset con datos de portátiles. 
</div>

<div style="border-style:groove;border-width:thin;padding:10px">
Debes hacer lo siguiente:
    <ol>
        <li>Carga el archivo "laptop_price.csv".</li>
        <li>Echa un vistazo a los datos. Corrige cosas si es necesario.</li>
        <li>Hay muchas columnas categóricas. Transfórmalas.</li>
        <li>Divide el dataset en conjuntos de training y de test.</li>
        <li>Soluciona el ejercicio con una red neuronal.</li>
    </ol>
</div>

In [2]:
# Datos y preprocesamiento
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Modelos de Clasificación
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Modelos de Regresión
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from tensorflow import keras

# Métricas de Clasificación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Métricas de Regresión
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print("Todas las librerías importadas correctamente")

I0000 00:00:1773940108.206016   14521 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773940117.821408   14521 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Todas las librerías importadas correctamente


In [3]:

df_prices = pd.read_csv('laptop_price.csv', encoding='latin-1')

df_prices.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [4]:
df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   Product           1303 non-null   object 
 3   TypeName          1303 non-null   object 
 4   Inches            1303 non-null   float64
 5   ScreenResolution  1303 non-null   object 
 6   Cpu               1303 non-null   object 
 7   Ram               1303 non-null   object 
 8   Memory            1303 non-null   object 
 9   Gpu               1303 non-null   object 
 10  OpSys             1303 non-null   object 
 11  Weight            1303 non-null   object 
 12  Price_euros       1303 non-null   float64
dtypes: float64(2), int64(1), object(10)
memory usage: 132.5+ KB


In [5]:
def see_columns_unique_values(df, columns):
    for column in columns:
        print(f"Columna: {column}")
        print(len(df[column].unique()))

see_columns_unique_values(df_prices, df_prices.columns)

Columna: laptop_ID
1303
Columna: Company
19
Columna: Product
618
Columna: TypeName
6
Columna: Inches
18
Columna: ScreenResolution
40
Columna: Cpu
118
Columna: Ram
9
Columna: Memory
39
Columna: Gpu
110
Columna: OpSys
9
Columna: Weight
179
Columna: Price_euros
791


In [6]:
df_prices['Gpu'].unique()
df_prices.columns

Index(['laptop_ID', 'Company', 'Product', 'TypeName', 'Inches',
       'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight',
       'Price_euros'],
      dtype='object')

In [7]:
df_prices.drop(columns=['laptop_ID', 'Product'], inplace=True)

In [8]:
for index, value in df_prices['Weight'].items():
    df_prices.at[index, 'Weight'] = float(value.replace('kg', '').strip())

In [9]:
for index, value in df_prices['Ram'].items():
    df_prices.at[index, 'Ram'] = int(value.replace('GB', '').strip())

In [10]:
df_prices['Ram'] = df_prices['Ram'].astype(int)
df_prices['Weight'] = df_prices['Weight'].astype(float)

In [11]:
df_prices.corr(numeric_only=True)['Price_euros'].abs().sort_values(ascending=False)[1:]

Ram       0.743007
Weight    0.210370
Inches    0.068197
Name: Price_euros, dtype: float64

In [12]:
df_prices = pd.get_dummies(df_prices, dtype=int)

In [13]:
df_prices.head()

,Inches,Ram,Weight,Price_euros,Company_Acer,Company_Apple,Company_Asus,Company_Chuwi,Company_Dell,Company_Fujitsu,...,Gpu_Nvidia Quadro M620M,OpSys_Android,OpSys_Chrome OS,OpSys_Linux,OpSys_Mac OS X,OpSys_No OS,OpSys_Windows 10,OpSys_Windows 10 S,OpSys_Windows 7,OpSys_macOS
0,13.3,8,1.37,1339.69,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,13.3,8,1.34,898.94,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,15.6,8,1.86,575.00,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,15.4,16,1.83,2537.45,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,13.3,8,1.37,1803.60,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [14]:
X = df_prices.drop(columns=['Price_euros'])
y = df_prices['Price_euros']

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [15]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
X, y, test_size=0.1, random_state=0)

X_train, X_valid, y_train, y_valid = train_test_split(
X_train_full, y_train_full, test_size=0.1, random_state=0)

In [16]:
model = keras.models.Sequential([
keras.layers.Dense(210, activation="relu", input_shape=X_train.shape[1:]),
keras.layers.Dense(140, activation="relu"),
keras.layers.Dense(1)
])

model.compile(loss="mean_absolute_error", optimizer="sgd",metrics=['mae'])


W0000 00:00:1773940124.879268   14521 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [17]:
history = model.fit(X_train, y_train, epochs=300,
validation_data=(X_valid, y_valid))

mse_test = model.evaluate(X_test, y_test)

Epoch 1/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1081.9449 - mae: 1081.9449 - val_loss: 1043.4652 - val_mae: 1043.4652
Epoch 2/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 535.2737 - mae: 535.2737 - val_loss: 289.4411 - val_mae: 289.4411
Epoch 3/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 266.8220 - mae: 266.8220 - val_loss: 222.2318 - val_mae: 222.2318
Epoch 4/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 217.8947 - mae: 217.8947 - val_loss: 276.3011 - val_mae: 276.3011
Epoch 5/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 198.7441 - mae: 198.7441 - val_loss: 235.5614 - val_mae: 235.5614
Epoch 6/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 181.3031 - mae: 181.3031 - val_loss: 254.2830 - val_mae: 254.2830
Epoch 7/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 181.9552 - mae: 181.9552 - val_loss: 216.5304 - val_mae: 216.5304
Epoch 8/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 192.3653 - mae: 192.3653 - val_loss: 218.7263 - val_mae: 218.7

In [18]:
X_new = X_test[:3] 
y_pred = model.predict(X_new)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


In [19]:
y_pred = model.predict(X_test)
r2_score(y_test, y_pred)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


0.7918798186353019

In [20]:
model.save("price_predictor.keras")